# PrimateFace Quick Start

| GitHub | Paper | Website | Models |
|---|---|---|---|
| [Code](https://github.com/KordingLab/PrimateFace) | [Preprint](https://www.biorxiv.org/content/10.1101/2025.08.12.669927) | [Project](https://primateface.studio/) | [HuggingFace](https://huggingface.co/fparodi/primateface-models) |

This notebook demonstrates the PrimateFace v2 API: detect primate faces, estimate 68-point facial landmarks, and extract analysis features — all in 3 lines of code.

**Requirements**: PrimateFace + MMDetection/MMPose + GPU (recommended)

## 1. Setup

In [ ]:
import sys
from pathlib import Path

# Ensure repo root is on path
NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from notebook_utils import check_environment, setup_publication_style
setup_publication_style()
env_info = check_environment(require_packages=["mmdet", "mmpose"])

## 2. Initialize PrimateFace

Models are automatically downloaded from [HuggingFace](https://huggingface.co/fparodi/primateface-models) on first use.

In [ ]:
from primateface import PrimateFace

pf = PrimateFace()  # auto-downloads models, uses GPU if available
print(f"Device: {pf.device}")
print(f"Pose model: {pf.pose_model}")

## 3. Analyze an Image

`.analyze()` runs face detection + 68-point landmark estimation and returns a list of `Face` objects.

In [ ]:
# Use the bundled test image
image_path = str(REPO_ROOT / "demos" / "ateles_000003.jpeg")

faces = pf.analyze(image_path)
print(f"Detected {len(faces)} face(s)")
for face in faces:
    print(face)

## 4. Inspect a Face Object

Each `Face` has detection outputs (bbox, score, keypoints) and lazy analysis properties that are computed on first access.

In [ ]:
face = faces[0]

# Detection outputs (always available)
print(f"Bounding box: {face.bbox}")
print(f"Confidence:   {face.score:.3f}")
print(f"Keypoints:    shape {face.keypoints.shape}  (68 landmarks x [x, y, score])")
print()

# Face crop
print(f"Crop shape:   {face.crop.shape}")

## 5. Analysis Features (Lazy Evaluation)

These are computed from landmarks on first access — no additional models needed.

In [ ]:
# Head pose (yaw, pitch, roll in degrees)
yaw, pitch, roll = face.head_pose
print(f"Head pose: yaw={yaw:.1f}, pitch={pitch:.1f}, roll={roll:.1f}")

# Facial symmetry (0 = perfect, higher = more asymmetric)
print(f"Symmetry (FA): {face.symmetry:.4f}")

# Kinematic features (~14 geometric measurements, normalized by IOD)
kin = face.kinematics
print(f"\nKinematic features ({len(kin)} total):")
for key, val in kin.items():
    print(f"  {key}: {val:.4f}")

# Shortcut properties
print(f"\nMouth aperture: {face.mouth_aperture:.4f}")
print(f"Eye aperture (R, L): {face.eye_aperture}")
print(f"Brow position (R, L): {face.brow_position}")
print(f"IOD (pixels): {face.interocular_distance:.1f}")

In [ ]:
# Face quality metrics
q = face.quality
print("Quality metrics:")
for key, val in q.items():
    print(f"  {key}: {val:.3f}")

# Per-region symmetry
rs = face.region_symmetry
print(f"\nRegion symmetry:")
for key, val in rs.items():
    print(f"  {key}: {val:.4f}")

## 6. Visualization

In [ ]:
import matplotlib.pyplot as plt
import cv2

# Draw detections + landmarks on the image
viz = PrimateFace.draw(faces, image_path)

# Display (convert BGR -> RGB for matplotlib)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

original = cv2.imread(image_path)
axes[0].imshow(cv2.cvtColor(original, cv2.COLOR_BGR2RGB))
axes[0].set_title("Original")
axes[0].axis("off")

axes[1].imshow(cv2.cvtColor(viz, cv2.COLOR_BGR2RGB))
axes[1].set_title(f"PrimateFace: {len(faces)} face(s)")
axes[1].axis("off")

plt.tight_layout()
plt.show()

## 7. Export Results

Use `primateface.io` to save results in common formats.

In [ ]:
from primateface.io import to_csv, to_coco_json

# Export to CSV (one row per face, all features)
to_csv(faces, "quickstart_results.csv", image_path=image_path)
print("Saved quickstart_results.csv")

# Export to COCO JSON
to_coco_json(faces, "quickstart_results.json", image_path=image_path)
print("Saved quickstart_results.json")

In [ ]:
# Preview the CSV
import pandas as pd
df = pd.read_csv("quickstart_results.csv")
print(f"CSV shape: {df.shape}")
df.head()

## 8. Pose Model Selection

PrimateFace supports multiple pose backends. The default `hrnet` is fast and lightweight (38 MB). For higher accuracy, use `vitpose` (1.2 GB).

In [ ]:
# To use ViTPose instead (downloads on first use):
# pf_vit = PrimateFace(pose_model="vitpose")
# faces_vit = pf_vit.analyze(image_path)

print("Available pose models: hrnet (default, 38 MB), vitpose (1.2 GB)")

## Summary

PrimateFace v2 provides:
- **3-line API**: `PrimateFace()` → `.analyze()` → `Face` objects
- **Auto model download** from HuggingFace Hub
- **68-point landmarks** with lazy analysis (head pose, symmetry, kinematics, quality)
- **Visualization** via `PrimateFace.draw()`
- **Export** to CSV, COCO JSON, and more via `primateface.io`
- **CLI**: `primateface analyze image.jpg`

See the other tutorials for in-depth applications:
- [Lemur Video Timestamping](lemur_video_timestamping.ipynb)
- [Macaque Face Recognition](macaque_face_recognition.ipynb)
- [Howler Vocal-Motor Coupling](howler_vocal_motor_coupling.ipynb)
- [Macaque Gaze Following](macaque_gaze_following.ipynb)
- [Landmark-Based Demographics](landmark_demographics.ipynb)